# Modelo LSTM con metodo por Transecto y metodo General

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
LSTM (Encoder-Decoder) con KerasTuner.
Verifica que validación no esté vacía.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed, Reshape, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import keras_tuner as kt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")
MODELS_DIR = os.path.join(BASE_DIR, "models")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)

DATA_DIR = WINDOWS_PARTITIONED_DIR
OUTPUT_DIR = os.path.join(MODELS_DIR, "lstm")
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)

HP_UNITS = [64, 128]          # elimina 32
HP_DROPOUT = [0.2, 0.3]       # elimina 0.0
HP_LR = [1e-3, 1e-4]          # elimina 5e-4
HP_EPOCHS = 100
BATCH_SIZE = 32


def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan


def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }


def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def build_model(hp, input_shape, output_dim=72):
    if isinstance(hp, dict):
        units = hp['units']
        dropout = hp['dropout']
        lr = hp['lr']
    else:
        units = hp.Choice('units', HP_UNITS)
        dropout = hp.Choice('dropout', HP_DROPOUT)
        lr = hp.Choice('lr', HP_LR)
    encoder_inputs = Input(shape=input_shape, name='encoder_input')
    encoder = LSTM(units, return_state=True, dropout=dropout, recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    _, state_h, state_c = encoder(encoder_inputs)
    encoder_states = [state_h, state_c]
    decoder_repeated = RepeatVector(output_dim)(state_h)
    decoder_lstm = LSTM(units, return_sequences=True, dropout=dropout, recurrent_dropout=dropout, kernel_regularizer=l2(1e-5))
    decoder_outputs = decoder_lstm(decoder_repeated, initial_state=encoder_states)
    decoder_dense = TimeDistributed(Dense(1))(decoder_outputs)
    outputs = Reshape((output_dim,))(decoder_dense)
    model = Model(encoder_inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model


def get_station_idx_and_mapping(entity_name, transect=True):
    if not transect:
        return None, None
    csv_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}.csv")
    mapping_path = os.path.join(ENCODED_DIR, "dl", "by_transect", f"{entity_name}_mapping.json")
    if not os.path.exists(csv_path):
        return None, None
    df_cols = pd.read_csv(csv_path, nrows=0, index_col=0)
    feature_names = df_cols.columns.tolist()
    if 'Estacion' not in feature_names:
        return None, None
    idx_estacion = feature_names.index('Estacion')
    if os.path.exists(mapping_path):
        with open(mapping_path, 'r') as f:
            mapping_data = json.load(f)
        est_map = mapping_data.get('Estacion', {})
        mapping_estacion = {int(v): k for k, v in est_map.items()}
    else:
        mapping_estacion = None
    return idx_estacion, mapping_estacion


def compute_per_station_metrics_lstm(y_true, y_pred, X_test, idx_estacion, mapping_estacion, output_dir, entity_name):
    station_preds = {}
    for i in range(len(y_true)):
        station_code = int(round(X_test[i, 0, idx_estacion]))
        station_name = mapping_estacion.get(station_code, f"Unknown_{station_code}")
        station_preds.setdefault(station_name, {'true': [], 'pred': []})
        station_preds[station_name]['true'].append(y_true[i])
        station_preds[station_name]['pred'].append(y_pred[i])
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])
        pred_stack = np.vstack(data['pred'])
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
    return station_metrics


def train_and_evaluate_lstm(X_train, y_train, X_val, X_test, y_val, y_test,
                            scaler_y, entity_name, output_subdir,
                            idx_estacion=None, mapping_estacion=None):
    print(f"\n--- Entrenando LSTM para {entity_name} ---")
    if len(X_val) == 0 or len(y_val) == 0:
        print(f"  Saltando {entity_name}: conjunto de validación vacío.")
        return None, None

    input_shape = (X_train.shape[1], X_train.shape[2])
    tuner = kt.RandomSearch(
        hypermodel=lambda hp: build_model(hp, input_shape),
        objective='val_loss',
        max_trials=len(HP_UNITS)*len(HP_DROPOUT)*len(HP_LR),
        executions_per_trial=1,
        directory=output_subdir,
        project_name='tuning',
        overwrite=True
    )
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    print("  Buscando hiperparámetros...")
    tuner.search(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=HP_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, reduce_lr], verbose=1)
    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_params = {'units': best_hp.get('units'), 'dropout': best_hp.get('dropout'), 'lr': best_hp.get('lr')}
    print(f"  Mejores parámetros: {best_params}")

    model = build_model(best_params, input_shape)
    X_train_full = np.concatenate([X_train, X_val], axis=0)
    y_train_full = np.concatenate([y_train, y_val], axis=0)
    history = model.fit(X_train_full, y_train_full, validation_split=0.1, epochs=HP_EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)], verbose=1)

    y_pred_scaled = model.predict(X_test, verbose=0)
    y_test_descaled = scaler_y.inverse_transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
    y_pred_descaled = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).reshape(y_pred_scaled.shape)
    test_metrics = compute_metrics(y_test_descaled.ravel(), y_pred_descaled.ravel())
    print(f"  Métricas test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}")

    if idx_estacion is not None and mapping_estacion is not None:
        station_metrics = compute_per_station_metrics_lstm(y_test_descaled, y_pred_descaled, X_test,
                                                           idx_estacion, mapping_estacion, output_subdir, entity_name)
    else:
        station_metrics = None

    model.save(os.path.join(output_subdir, "model.keras"))
    with open(os.path.join(output_subdir, "history.pkl"), 'wb') as f:
        pickle.dump(history.history, f)
    results = {'best_params': best_params, 'test_metrics': test_metrics, 'station_metrics': station_metrics,
               'n_train': len(X_train_full), 'n_test': len(X_test)}
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    horizons = [23, 47, 71]
    plot_predictions(y_test_descaled, y_pred_descaled, horizons,
                     os.path.join(output_subdir, "test_scatter.png"), f"LSTM - {entity_name} - Test")
    np.save(os.path.join(output_subdir, "test_pred.npy"), y_pred_descaled)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test_descaled)
    return model, test_metrics


def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO LSTM POR TRANSECTO")
    dl_dir = os.path.join(DATA_DIR, "by_transect", "dl")
    if not os.path.exists(dl_dir):
        return
    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val   = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val   = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test  = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test  = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)
        idx_estacion, mapping_estacion = get_station_idx_and_mapping(entity, transect=True)
        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_lstm(X_train, y_train, X_val, X_test, y_val, y_test,
                                scaler_y, entity, out_subdir, idx_estacion, mapping_estacion)


def process_global():
    print("\n" + "="*50)
    print("PROCESANDO LSTM GLOBAL")
    dl_dir = os.path.join(DATA_DIR, "global", "dl")
    if not os.path.exists(dl_dir):
        return
    entities = [d for d in os.listdir(dl_dir) if os.path.isdir(os.path.join(dl_dir, d))]
    for entity in entities:
        entity_path = os.path.join(dl_dir, entity)
        X_train = np.load(os.path.join(entity_path, "train_X.npy"))
        y_train = np.load(os.path.join(entity_path, "train_y.npy"))
        X_val   = np.load(os.path.join(entity_path, "val_X.npy"))
        y_val   = np.load(os.path.join(entity_path, "val_y.npy"))
        X_test  = np.load(os.path.join(entity_path, "test_X.npy"))
        y_test  = np.load(os.path.join(entity_path, "test_y.npy"))
        with open(os.path.join(entity_path, "scaler_y.pkl"), 'rb') as f:
            scaler_y = pickle.load(f)
        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)
        train_and_evaluate_lstm(X_train, y_train, X_val, X_test, y_val, y_test,
                                scaler_y, entity, out_subdir, None, None)


def generate_summary():
    summary = []
    for t in ["by_transect", "global"]:
        dir_path = os.path.join(OUTPUT_DIR, t)
        if not os.path.exists(dir_path):
            continue
        for entity in os.listdir(dir_path):
            res_file = os.path.join(dir_path, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({'entity': entity, 'type': t, **data['test_metrics']})
    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado.")


if __name__ == "__main__":
    print("LSTM")
    process_by_transect()
    process_global()
    generate_summary()

Trial 7 Complete [00h 13m 14s]
val_loss: 0.016640836372971535

Best val_loss So Far: 0.016640836372971535
Total elapsed time: 02h 01m 04s
  Mejores parámetros: {'units': 128, 'dropout': 0.2, 'lr': 0.001}
Epoch 1/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 43s 141ms/step - loss: 0.0314 - mae: 0.1369 - val_loss: 0.0224 - val_mae: 0.1183 - learning_rate: 0.0010
Epoch 2/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 34s 136ms/step - loss: 0.0248 - mae: 0.1245 - val_loss: 0.0220 - val_mae: 0.1176 - learning_rate: 0.0010
Epoch 3/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 34s 136ms/step - loss: 0.0226 - mae: 0.1180 - val_loss: 0.0206 - val_mae: 0.1119 - learning_rate: 0.0010
Epoch 4/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 34s 137ms/step - loss: 0.0162 - mae: 0.0960 - val_loss: 0.0182 - val_mae: 0.1047 - learning_rate: 0.0010
Epoch 5/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 34s 136ms/step - loss: 0.0147 - mae: 0.0910 - val_loss: 0.0176 - val_mae: 0.1028 - learning_rate: 0.0010
Epoch 6/100
248/248 ━━━━━━━━━━━━━━━━━━━━ 34s 136ms/step - loss: 0.

# Grafico serie temporal O3 + Predicciones

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Gráficas de predicción de O₃ con LSTM para múltiples estaciones.
Para cada estación:
- Muestra los últimos 7 días observados de O₃ (valores originales).
- Predice los siguientes 3 días (72h) usando el modelo LSTM entrenado.
Se aplica la misma normalización (MinMaxScaler) usada durante el entrenamiento.
Las gráficas se guardan en una carpeta 'lstm_forecasts'.
"""

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model

# =============================================================================
# CONFIGURACIÓN (AJUSTAR SEGÚN TU ESTRUCTURA DE DIRECTORIOS)
# =============================================================================
# Lista de estaciones a procesar
STATIONS = [
    "T1_E1_Alicante",
    "T1_E2_Elda",
    "T2_E1_Elche",
    "T2_E2_Elda",
    "T3_E1_Valencia",
    "T3_E2_Buñol",
    "T4_E1_Valencia",
    "T4_E2_Villar_Arzobispo",
    "T5_E1_Castellon",
    "T5_E2_Onda",
    "T6_E1_Sant_Jordi",
    "T6_E2_Coratxa",
    "T6_E3_Zorita",
    "T8_E1_Sant_Jordi",
    "T8_E2_Morella",
    "T8_E3_Zorita"
]

# Rutas base (cambiar según tu sistema)
BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
MODELS_BASE = os.path.join(BASE_DIR, "models", "lstm", "global")   # dentro de cada estación hay model.keras
SCALERS_BASE = os.path.join(BASE_DIR, "windows_partitioned", "global", "dl")  # contiene scaler_X.pkl y scaler_y.pkl
ENCODED_DIR = os.path.join(BASE_DIR, "encoded", "ml", "global")   # archivos CSV con datos originales
OUTPUT_DIR = os.path.join(BASE_DIR, "lstm_forecasts")              # carpeta donde guardar las gráficas

# Parámetros de ventana (deben coincidir con el entrenamiento)
WINDOW_IN = 72      # horas de entrada
WINDOW_OUT = 72     # horas de salida (predicción)

# =============================================================================
# FUNCIONES
# =============================================================================
def load_original_data(station):
    """
    Carga el CSV original con todas las variables horarias para una estación.
    El CSV debe tener un índice datetime (columna 'timestamp' o índice temporal).
    Retorna un DataFrame con frecuencias horarias completas.
    """
    csv_path = os.path.join(ENCODED_DIR, f"{station}.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
    # Asegurar frecuencia horaria (interpolación opcional para rellenar NaN)
    df = df.asfreq('H')
    # Interpolar valores faltantes (lineal en el tiempo)
    df = df.interpolate(method='time', limit_direction='both')
    return df

def get_last_window(df, window_size=WINDOW_IN):
    """
    Extrae las últimas `window_size` filas del DataFrame (todas las variables).
    Si hay menos filas, lanza error.
    Retorna un array 2D de forma (window_size, n_features).
    """
    if len(df) < window_size:
        raise ValueError(f"Solo hay {len(df)} registros, se necesitan {window_size}")
    window = df.iloc[-window_size:].values  # (window_size, n_features)
    return window

def normalize_window(window, scaler_X):
    """
    Normaliza la ventana de entrada usando el scaler_X entrenado.
    window: array 2D de forma (window_size, n_features)
    Retorna array normalizado con la misma forma pero agregando dimensión de batch: (1, window_size, n_features)
    """
    n_timesteps, n_features = window.shape
    window_norm = scaler_X.transform(window)  # (window_size, n_features)
    return window_norm.reshape(1, n_timesteps, n_features)  # (1, window_size, n_features)

def predict_next_72h(model, norm_window):
    """
    Usa el modelo LSTM para predecir los siguientes 72 valores de O₃.
    norm_window: array 3D de forma (1, WINDOW_IN, n_features) ya normalizado.
    Retorna array de predicciones normalizadas de longitud WINDOW_OUT.
    """
    pred_norm = model.predict(norm_window, verbose=0)  # forma (1, 72)
    if pred_norm.ndim == 2:
        pred_norm = pred_norm.flatten()
    return pred_norm

def plot_and_save_forecast(station, historical_o3, forecast, forecast_start_date, output_dir):
    """
    Genera la gráfica y la guarda en output_dir.
    historical_o3: Serie de pandas con índice datetime (últimos 7 días de O3 original)
    forecast: array de 72 valores predichos (en unidades originales)
    forecast_start_date: datetime del primer valor predicho
    """
    forecast_index = pd.date_range(start=forecast_start_date, periods=len(forecast), freq='H')
    
    plt.figure(figsize=(14, 5))
    plt.plot(historical_o3.index, historical_o3.values, 'b-', linewidth=1.5, label='Observado (última semana)')
    plt.plot(forecast_index, forecast, 'r--', linewidth=2, label='Predicción LSTM (72h)')
    plt.axvline(x=historical_o3.index[-1], color='gray', linestyle=':', label='Inicio predicción')
    plt.title(f"Predicción de O₃ con LSTM - {station}\nÚltima semana observada + 3 días pronosticados")
    plt.xlabel("Fecha y hora")
    plt.ylabel("Concentración de O₃ (µg/m³)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"{station}_forecast.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Gráfica guardada: {out_path}")

# =============================================================================
# PROGRAMA PRINCIPAL
# =============================================================================
if __name__ == "__main__":
    print("Generando predicciones para múltiples estaciones con LSTM...")
    print(f"Directorio de salida: {OUTPUT_DIR}")
    
    for station in STATIONS:
        print(f"\n--- Procesando {station} ---")
        
        # 1. Ruta del modelo y los scalers
        model_path = os.path.join(MODELS_BASE, station, "model.keras")
        scaler_X_path = os.path.join(SCALERS_BASE, station, "scaler_X.pkl")
        scaler_y_path = os.path.join(SCALERS_BASE, station, "scaler_y.pkl")
        
        if not os.path.exists(model_path):
            print(f"  Modelo no encontrado en {model_path}. Se omite.")
            continue
        if not os.path.exists(scaler_X_path):
            print(f"  scaler_X no encontrado en {scaler_X_path}. Se omite.")
            continue
        if not os.path.exists(scaler_y_path):
            print(f"  scaler_y no encontrado en {scaler_y_path}. Se omite.")
            continue
        
        # 2. Cargar modelo y scalers
        try:
            model = load_model(model_path)
            print(f"  Modelo LSTM cargado correctamente.")
            with open(scaler_X_path, 'rb') as f:
                scaler_X = pickle.load(f)
            with open(scaler_y_path, 'rb') as f:
                scaler_y = pickle.load(f)
            print(f"  Scalers cargados.")
        except Exception as e:
            print(f"  Error al cargar modelo/scalers: {e}")
            continue
        
        # 3. Cargar datos originales de la estación
        try:
            df = load_original_data(station)
            print(f"  Datos cargados: {len(df)} registros, {df.shape[1]} variables.")
        except Exception as e:
            print(f"  Error al cargar datos CSV: {e}")
            continue
        
        # 4. Verificar columna 'O3'
        if 'O3' not in df.columns:
            print(f"  El CSV no contiene columna 'O3'. Se omite.")
            continue
        
        # 5. Extraer última ventana de entrada (72h de todas las variables)
        try:
            last_window = get_last_window(df, WINDOW_IN)
            print(f"  Ventana de entrada extraída con forma {last_window.shape}")
        except Exception as e:
            print(f"  Error al extraer ventana: {e}")
            continue
        
        # 6. Normalizar la ventana con scaler_X
        try:
            norm_input = normalize_window(last_window, scaler_X)
            print(f"  Entrada normalizada con forma {norm_input.shape}")
        except Exception as e:
            print(f"  Error al normalizar ventana: {e}")
            continue
        
        # 7. Realizar predicción (salida normalizada)
        try:
            pred_norm = predict_next_72h(model, norm_input)
            print(f"  Predicción normalizada generada.")
        except Exception as e:
            print(f"  Error en predicción: {e}")
            continue
        
        # 8. Desescalar la predicción a unidades originales
        try:
            pred_original = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
            print(f"  Predicción desescalada: {len(pred_original)} valores")
        except Exception as e:
            print(f"  Error al desescalar predicción: {e}")
            continue
        
        # 9. Fecha de inicio del pronóstico (hora siguiente a la última observada)
        last_observed_time = df.index[-1] + pd.Timedelta(hours=1)
        
        # 10. Obtener últimos 7 días de O3 observado (168 horas) - valores originales
        last_week_o3 = df['O3'].iloc[-168:]
        
        # 11. Generar y guardar gráfica
        try:
            plot_and_save_forecast(station, last_week_o3, pred_original, last_observed_time, OUTPUT_DIR)
        except Exception as e:
            print(f"  Error al generar gráfica: {e}")
    
    print("\nProceso completado.")

In [2]:
import numpy as np
from pathlib import Path

base = "/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/windows/by_transect/ml"
for f in Path(base).glob("*_X.npy"):
    name = f.stem.replace("_X", "")
    X = np.load(f)
    y = np.load(f.parent / f"{name}_y.npy")
    print(f"{name}: X {X.shape}, y {y.shape}  -> {X.shape[0] == y.shape[0]}")

Transecto_1: X (17301, 1728), y (17301, 72)  -> True
Transecto_2: X (16798, 1728), y (16798, 72)  -> True


In [3]:
import numpy as np
from pathlib import Path

base = "/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/windows_partitioned/by_transect/dl"
for entity in Path(base).iterdir():
    if entity.is_dir():
        train_X = np.load(entity / "train_X.npy")
        train_y = np.load(entity / "train_y.npy")
        val_X   = np.load(entity / "val_X.npy")
        val_y   = np.load(entity / "val_y.npy")
        test_X  = np.load(entity / "test_X.npy")
        test_y  = np.load(entity / "test_y.npy")
        print(f"{entity.name}: train {train_X.shape[0]} == {train_y.shape[0]}? {train_X.shape[0]==train_y.shape[0]}")
        print(f"         val   {val_X.shape[0]} == {val_y.shape[0]}? {val_X.shape[0]==val_y.shape[0]}")
        print(f"         test  {test_X.shape[0]} == {test_y.shape[0]}? {test_X.shape[0]==test_y.shape[0]}")

Transecto_1: train 8698 == 749? False
         val   7807 == 8698? False
         test  749 == 7807? False
Transecto_2: train 8399 == 687? False
         val   7665 == 8399? False
         test  687 == 7665? False
